### We need to access the hugging face token in order to use LLaMa2 model. If you are a google colab user then put the following command and make sure you have been granted for accessing the LLaMa2 model. You can request an access permission if you dont have it already: https://huggingface.co/meta-llama/Llama-2-13b-chat-hf

In [ ]:
! huggingface-cli login

In [ ]:
!pip install huggingface_hub

  Using cached huggingface_hub-0.29.3-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.29.3-py3-none-any.whl (468 kB)


### If you are notebook user, then you have to put the following command in order to get hugging face credentials.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install pandas

  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)
Using cached tzdata-2025.1-py2.py3-none-any.whl (346 kB)


In [ ]:
!pip install transformers

  Using cached regex-2024.11.6-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.0 MB 5.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/10.0 MB 5.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/10.0 MB 4.5 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/10.0 MB 4.1 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/10.0 MB 3.8 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/10.0 MB 3.6 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.0 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/10.0 MB 3.4 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/10.0 MB 3.4 MB/s eta 0:00:02
   --------------------------- ------------ 6.8/10.0 MB 3.3 MB/s eta 0:00:01
   --------

In [ ]:
pip install --upgrade accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import accelerate
print(accelerate.__version__)

1.5.2


In [ ]:
pip install --upgrade torch torchvision torchaudio transformers

Note: you may need to restart the kernel to use updated packages.


### Now, run the following script and put the PRESCRIPTIONS.csv file to any folder.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# ✅ Load MIMIC-III Dataset (Modify Path as Needed)
df = pd.read_csv("C:\\Users\\Abu Sufian Rupok\\Desktop\\PRESCRIPTIONS.csv", low_memory=False)  # Fix warning

# ✅ Extract Clinical Notes or Relevant Text Fields
questions = df["DRUG_NAME_GENERIC"].dropna().astype(str).tolist()
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define LLaMA-2 Models
LLAMA_MODELS = {
    "LLaMA-2-7B": "meta-llama/Llama-2-7b-hf",
    "LLaMA-2-13B": "meta-llama/Llama-2-13b-hf"
}

# ✅ Check if GPU is Available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ✅ Bayesian LLaMA Class (MC Dropout)
class BayesianLLaMA(nn.Module):
    def __init__(self, base_model):
        super(BayesianLLaMA, self).__init__()
        self.base_model = base_model
        self.device = next(base_model.parameters()).device  # Fix device issue

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        return logits

# ✅ Function: Generate Response (No Uncertainty)
def generate_response_no_uncertainty(model, tokenizer, input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}  # Move inputs to correct device

    with torch.no_grad():
        output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)

    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# ✅ Function: Bayesian Uncertainty Estimation (MC Dropout)
def estimate_uncertainty_llama(model, tokenizer, input_text, n_samples=10):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(model.device) for key, val in inputs.items()}  # Fix: Move inputs to correct device

    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = model(inputs["input_ids"], inputs["attention_mask"])
            logits = nn.functional.dropout(logits, p=0.3, training=True)  # Bayesian Dropout
            predictions.append(logits.cpu().numpy())  # Move logits to CPU for processing

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty
    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Run Evaluation for LLaMA-2 (7B & 13B) on GPU
results = {}

for model_name, model_path in LLAMA_MODELS.items():
    print(f"\n====== Running {model_name} on MIMIC-III (GPU) ======")

    # ✅ Load Tokenizer & Model (Ensure GPU Execution)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token  # Fix tokenizer warning
    tokenizer.padding_side = "left"  # Needed for some causal LLMs

    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.float16,  # Use FP16 for better GPU performance
        device_map="auto"  # Automatically allocate model layers to GPU
    )

    bayesian_model = BayesianLLaMA(model)  # Fix: No need to call .to(device)

    cus_scores_no_uncertainty = []
    cus_scores_with_uncertainty = []

    for i in range(num_samples):
        # 🔹 Without Uncertainty
        response_no_uncertainty = generate_response_no_uncertainty(model, tokenizer, questions[i])
        print(f"Sample {i+1} - {model_name} Response (No Uncertainty): {response_no_uncertainty}")

        cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
        cus_scores_no_uncertainty.append(cus_score_no_uncertainty)

        # 🔹 With Bayesian Uncertainty Modeling
        mean_pred, uncertainty = estimate_uncertainty_llama(bayesian_model, tokenizer, questions[i])
        cus_score = compute_cus([uncertainty])
        cus_scores_with_uncertainty.append(cus_score)

        print(f"Sample {i+1} - {model_name} CUS (With Uncertainty): {cus_score}")

    # ✅ Compute Overall ZTI
    zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
    zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)

    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

    # ✅ Store Results
    results[model_name] = {
        "CUS_No_Uncertainty": cus_scores_no_uncertainty,
        "CUS_With_Uncertainty": cus_scores_with_uncertainty,
        "ZTI_No_Uncertainty": zti_score_no_uncertainty,
        "ZTI_With_Uncertainty": zti_score_with_uncertainty,
    }

# ✅ Final Comparison Report
print("\n===== Final Comparison Report =====")
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
    print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
    print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
    print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])

Using device: cuda

====== Running LLaMA-2-7B on MIMIC-III (GPU) ======


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
D:\Anaconda\Lib\site-packages\transformers\generation\utils.py:2105: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cuda, whereas the model is on cpu. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cpu') before running `.generate()`.
  warnings.warn(


Sample 1 - LLaMA-2-7B Response (No Uncertainty): Tacrolimus (Prograf, Protopic)
Prograf (tacrolimus) is a drug used to prevent rejection of organ transplants. Prograf is also used to treat atopic dermatitis (eczema).
Sample 1 - LLaMA-2-7B CUS (With Uncertainty): 0.662109375
Sample 2 - LLaMA-2-7B Response (No Uncertainty): Warfarin, a blood-thinning medication, is prescribed to prevent blood clots in people with atrial fibrillation (AFib). This is a condition that causes an irregular heartbeat.
Atrial fibrill
Sample 2 - LLaMA-2-7B CUS (With Uncertainty): 0.66015625
Sample 3 - LLaMA-2-7B Response (No Uncertainty): Furosemide is a diuretic used to treat fluid retention in people with congestive heart failure, liver disease, or a kidney disorder such as nephrotic syndrome.
Furosemide is a diure
Sample 3 - LLaMA-2-7B CUS (With Uncertainty): 0.6611328125
Sample 4 - LLaMA-2-7B Response (No Uncertainty): Warfarin is a prescription medicine that is used to prevent and treat blood clots. It is a

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abu Sufian Rupok\.cache\huggingface\hub\models--meta-llama--Llama-2-13b-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


... (8 outputs left out)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Sample 1 - LLaMA-2-13B Response (No Uncertainty): Tacrolimus (FK506) is a potent immunosuppressant drug that is used to prevent rejection of organ transplants. Tacrolimus is a macrolide lactone that binds to an immunoph
Sample 1 - LLaMA-2-13B CUS (With Uncertainty): 0.66064453125
Sample 2 - LLaMA-2-13B Response (No Uncertainty): Warfarin and the risk of thrombosis in patients with diabetes
G. G. Gross, P. C. Hsu, S. J. Kimmel
The risk of thrombosis in diabetic patients on
Sample 2 - LLaMA-2-13B CUS (With Uncertainty): 0.66064453125
Sample 3 - LLaMA-2-13B Response (No Uncertainty): Furosemide (Lasix) is a loop diuretic that prevents your body from absorbing too much salt, allowing the salt to instead be passed in your urine. It is used to treat fluid retention (edema) in people with
Sample 3 - LLaMA-2-13B CUS (With Uncertainty): 0.66064453125
Sample 4 - LLaMA-2-13B Response (No Uncertainty): Warfarin is a medication used to prevent blood clots. It is used as an anticoagulant. Warfarin i